In this script we will develop a basic training pipeline and try to develop the maximum functionality manually with <b>no nn.Modeule</b>

In [6]:
import torch
from sklearn.model_selection import train_test_split

In [7]:
data = torch.rand(10,5)
data

tensor([[0.0870, 0.6330, 0.0236, 0.0734, 0.3905],
        [0.4308, 0.6014, 0.2455, 0.6606, 0.0966],
        [0.0728, 0.2543, 0.2383, 0.4020, 0.8478],
        [0.9163, 0.0498, 0.6864, 0.2137, 0.8773],
        [0.0540, 0.6623, 0.1877, 0.4642, 0.4645],
        [0.2978, 0.4336, 0.6448, 0.4552, 0.8567],
        [0.6530, 0.5220, 0.8953, 0.2192, 0.3481],
        [0.1983, 0.1976, 0.3735, 0.9865, 0.1929],
        [0.6269, 0.9744, 0.6814, 0.0948, 0.1310],
        [0.3147, 0.9649, 0.9094, 0.8389, 0.2186]])

In [8]:
x = data
y = torch.tensor([1,0,1,0,1,0,1,0,1,0], dtype=torch.float32) #data[:,-1]

four features and one target

In [9]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [10]:
class BasicANN():
    def __init__(self,x):
        # requires_grad True means that we want to calculate the gradients for these parameters during backpropagation
        self.weights = torch.rand(x.shape[1],1,requires_grad=True)
        self.bias = torch.rand(1,requires_grad=True)

    def forward(self,x):
        # forward pass we will calculate the linear transformation of the input data and then apply the sigmoid activation 
        # function to get the predicted output
        # z = activation_function(x*weights + bias)
        z = torch.matmul(x,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss(self,y_pred,y):
        # to avoid log(0) which is undefined we will clamp the predicted values between a small value (epsilon) and 1-epsilon
        # this will ensure that the predicted values are always between epsilon and 1-epsilon, preventing any issues with log(0)
        # binary cross entropy loss is calculated as -y*log(y_pred) - (1-y)*log(1-y_pred) and then we take the mean of the loss 
        # values to get the average loss over the batch
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)
        loss = -y*torch.log(y_pred) - (1-y)*torch.log(1-y_pred) # binary cross endtropy
        return loss.mean()

In [11]:
# parameters of the model are the weights and bias which we want to optimize during training
learning_rate = 0.01
epochs = 10

In [12]:
# create an instance of the BasicANN class
model = BasicANN(X_train)

In [13]:
print('Weights:', model.weights)
print('Bias:', model.bias)

Weights: tensor([[0.2949],
        [0.3314],
        [0.7117],
        [0.4241],
        [0.9258]], requires_grad=True)
Bias: tensor([0.8586], requires_grad=True)


In [14]:
# ------ 1 Epoch -------
# 1. forward propagation
# 2. calculate loss
# 3. backward propagation
# 4. update parameters

In [15]:
# 1. forward propagation
y_pred = model.forward(X_train)
# 2. calculate loss
loss = model.loss(y_pred,y_train)
print('Loss:', loss.item())
# 3. backward propagation
loss.backward()
# 4. update parameters
with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad
    # zero the gradients after updating
    model.weights.grad.zero_()
    model.bias.grad.zero_()

Loss: 1.1874178647994995


In [16]:
for epoch in range(epochs):
    # 1. forward propagation
    y_pred = model.forward(X_train)
    # 2. calculate loss
    loss = model.loss(y_pred,y_train)
    print(f'# Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    # 3. backward propagation
    loss.backward()
    # 4. update parameters
    with torch.no_grad(): 
        # we use torch.no_grad() to disable gradient tracking during the parameter update step, 
        # since we don't want to calculate gradients for the parameters during this step
        # weights_new = weights_old - learning_rate * weights_old.grad # d(loss)/d(weights_old)
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
        # zero the gradients after updating
        model.weights.grad.zero_()
        model.bias.grad.zero_()

# Epoch 1/10, Loss: 1.1842
# Epoch 2/10, Loss: 1.1811
# Epoch 3/10, Loss: 1.1779
# Epoch 4/10, Loss: 1.1747
# Epoch 5/10, Loss: 1.1716
# Epoch 6/10, Loss: 1.1685
# Epoch 7/10, Loss: 1.1654
# Epoch 8/10, Loss: 1.1623
# Epoch 9/10, Loss: 1.1592
# Epoch 10/10, Loss: 1.1561
